In [1]:
import pandas as pd
import numpy as np
import re

df = pd.read_excel('lava.xlsx')

df['MRP'] = df['MRP'].str.replace('₹', '').str.replace(' ', '').str.replace(',', '').astype(float)
df['Current Price'] = df['Current Price'].str.replace('₹', '').str.replace(',', '').astype(float)

def extract_discount(value):
    if pd.isna(value):
        return 0  # or return None
    return int(value.replace('% off', ''))
df['Discount_Number'] = df['Discount'].apply(extract_discount)

df.drop("Discount", axis=1, inplace=True)

brands = {
    'Apple': ['apple', 'iphone', 'ipad'],
    'motorola': ['motorola', 'moto'],
    'AI_plus': ['ai+', 'ai', 'aiplus', 'ai plus', 'a i'], 
    'IQOO': ['iqoo'],
    'Tecno': ['tecno'],
    'lava': ['lava'],
    'readmi': ['readmi', 'redmi'],  
    'realme': ['realme'],
    'POCO': ['poco'],
    'nothing': ['nothing'],
    'Infix': ['infix'],
    'Oppo': ['oppo'],
    'vivo': ['vivo'],
    'Google': ['google', 'pixel'],
    'OnePlus': ['oneplus', 'one plus'],
    'Samsung': ['samsung', 'galaxy'],
}

def get_brand(product_name):
    if pd.isna(product_name):
        return 'Unknown'
    
    product_name = str(product_name).lower()
    
    for brand, keywords in brands.items():
        for keyword in keywords:
            if keyword in product_name:
                return brand
    return 'Unknown'
df['Brand'] = df['Product Name'].apply(get_brand)

df['RAM'] = df['RAM'].str.extract(r'(\d+)').astype(int)


df['Storage'] = df['Storage'].str.extract(r'(\d+)').astype(int)


def extract_ratings_reviews(text):
    """
    Extract ratings and reviews from text
    Example: "2,46,375 Ratings & 9,265 Reviews"
    Returns: (ratings, reviews)
    """
    if pd.isna(text):
        return None, None
    
    text = str(text)
    
    # Extract ratings (numbers before 'Ratings')
    ratings_match = re.search(r'([\d,]+)\s*Ratings', text, re.IGNORECASE)
    ratings = ratings_match.group(1).replace(',', '') if ratings_match else None
    
    # Extract reviews (numbers before 'Reviews')
    reviews_match = re.search(r'([\d,]+)\s*Reviews', text, re.IGNORECASE)
    reviews = reviews_match.group(1).replace(',', '') if reviews_match else None
    
    return ratings, reviews

# Apply to dataframe
df[['Ratings', 'Reviews']] = df['Review text'].apply(
    lambda x: pd.Series(extract_ratings_reviews(x))
)

# Convert to numbers
df['Ratings'] = pd.to_numeric(df['Ratings'], errors='coerce')
df['Reviews'] = pd.to_numeric(df['Reviews'], errors='coerce')


def extract_battery(text):
    if pd.isna(text):
        return 0
    match = re.search(r'(\d+)', str(text))
    return int(match.group()) if match else 0

df['Battery'] = df['Battery'].apply(extract_battery)

In [2]:
df.head()

,Product Name,Current Price,MRP,Rating,Review text,RAM,Storage,Display,Camera,Battery,Processor,Product URL,Image URL,Discount_Number,Brand,Ratings,Reviews
0,"LAVA Agni 3 5G (Pristine Glass, 128 GB)",21499.0,29999.0,3.8,374 Ratings & 38 Reviews,8,128,17.02 cm (6.7 inch) Full HD+ Display,50MP Rear Camera,5000,NaN,https://www.flipkart.com/lava-agni-3-5g-pristi...,https://rukminim2.flixcart.com/image/312/312/x...,28,lava,374.0,38.0
1,"LAVA Play Ultra 5G (Arctic Frost, 128 GB)",19499.0,23999.0,4.1,14 Ratings & 2 Reviews,8,128,16.76 cm (6.6 inch) Display,64MP Rear Camera,5000,NaN,https://www.flipkart.com/lava-play-ultra-5g-ar...,https://rukminim2.flixcart.com/image/312/312/x...,18,lava,14.0,2.0
2,"LAVA YUVA STAR 2 (SPARKLING IVORY, 64 MB)",10699.0,10999.0,2.6,15 Ratings & 0 Reviews,4,64,17.14 cm (6.75 inch) Display,13MP Rear Camera,5000,NaN,https://www.flipkart.com/lava-yuva-star-2-spar...,https://rukminim2.flixcart.com/image/312/312/x...,2,lava,15.0,0.0
3,"LAVA Shark 2 (Eclipse Grey, 64 GB)",12900.0,12999.0,3.5,26 Ratings & 1 Reviews,4,64,16.94 cm (6.67 inch) HD+ Display,NaN,5000,UNISOC T606 octa-core processor Processor,https://www.flipkart.com/lava-shark-2-eclipse-...,https://rukminim2.flixcart.com/image/312/312/x...,0,lava,26.0,1.0
4,"LAVA Shark 5G (stellar gold /steller gold, 128...",13999.0,14999.0,4.1,460 Ratings & 31 Reviews,4,128,17.02 cm (6.7 inch) HD+ Display,13MP + 13MP | 5MP + 5MP Dual Front Camera,5000,OCTA CORE UNISOC T765 Processor,https://www.flipkart.com/lava-shark-5g-stellar...,https://rukminim2.flixcart.com/image/312/312/x...,6,lava,460.0,31.0


In [3]:
df.isnull().sum()

Product Name         0
Current Price        0
MRP                156
Rating               6
Review text          6
RAM                  0
Storage              0
Display              0
Camera               6
Battery              0
Processor          150
Product URL          0
Image URL            0
Discount_Number      0
Brand                0
Ratings              6
Reviews              6
dtype: int64

In [4]:
df['Rating'] = df['Rating'].fillna(df['Rating'].median())
df['Ratings'] = df['Ratings'].fillna(df['Ratings'].median()).astype(int)
df['Reviews'] = df['Reviews'].fillna(df['Reviews'].median()).astype(int)

df['MRP'] = np.where(
    (df['Discount_Number'] == 0) & (df['MRP'].isna() | (df['MRP'] == '')),
    df['Current Price'],
    df['MRP']
)

In [5]:
df.isnull().sum()

Product Name         0
Current Price        0
MRP                  0
Rating               0
Review text          6
RAM                  0
Storage              0
Display              0
Camera               6
Battery              0
Processor          150
Product URL          0
Image URL            0
Discount_Number      0
Brand                0
Ratings              0
Reviews              0
dtype: int64

In [6]:
def get_lava_processor(product_name):
    if pd.isna(product_name):
        return 'Unknown Processor'
    
    # Blaze Series
    if 'Blaze 5G' in product_name:
        return 'MediaTek Dimensity 700'
    elif 'Blaze Pro' in product_name:
        return 'MediaTek Helio G88'
    elif 'Blaze 2' in product_name:
        return 'MediaTek Helio G37'
    elif 'Blaze 1' in product_name:
        return 'MediaTek Helio G37'
    
    # Yuva Series
    elif 'Yuva 5G' in product_name:
        return 'MediaTek Dimensity 700'
    elif 'Yuva 3' in product_name:
        return 'Unisoc T606'
    elif 'Yuva 2' in product_name:
        return 'MediaTek Helio G37'
    elif 'Yuva Pro' in product_name:
        return 'MediaTek Helio G85'
    elif 'Yuva' in product_name:
        return 'MediaTek Helio G37'
    
    # Agni Series
    elif 'Agni 5G' in product_name:
        return 'MediaTek Dimensity 810'
    elif 'Agni 2' in product_name:
        return 'MediaTek Dimensity 7050'
    elif 'Agni' in product_name:
        return 'MediaTek Dimensity 810'
    
    # Z Series
    elif 'Z' in product_name:
        if 'Z1' in product_name or 'Z2' in product_name:
            return 'MediaTek Helio G37'
        else:
            return 'MediaTek Helio G37'
    
    # A Series
    elif 'A' in product_name:
        if 'A9' in product_name:
            return 'MediaTek Helio G37'
        else:
            return 'MediaTek Helio G37'
    
    # B Series
    elif 'B' in product_name:
        return 'MediaTek Helio G37'
    
    # C Series
    elif 'C' in product_name:
        return 'MediaTek Helio G37'
    
    else:
        return 'Unknown Processor'

df.loc[df['Processor'].isnull(), 'Processor'] = df.loc[df['Processor'].isnull(), 'Product Name'].apply(get_lava_processor)

In [7]:
df.isnull().sum()

Product Name       0
Current Price      0
MRP                0
Rating             0
Review text        6
RAM                0
Storage            0
Display            0
Camera             6
Battery            0
Processor          0
Product URL        0
Image URL          0
Discount_Number    0
Brand              0
Ratings            0
Reviews            0
dtype: int64

In [8]:
def get_lava_camera(product_name):
    if pd.isna(product_name):
        return 'Unknown Camera'
    
    # Blaze Series
    if 'Blaze 5G' in product_name:
        return '50MP + 2MP + 2MP'
    elif 'Blaze Pro' in product_name:
        return '50MP + 2MP + 2MP'
    elif 'Blaze 2' in product_name:
        return '13MP + 2MP'
    elif 'Blaze 1' in product_name:
        return '13MP + 2MP'
    
    # Yuva Series
    elif 'Yuva 5G' in product_name:
        return '50MP + 2MP + 2MP'
    elif 'Yuva 3' in product_name:
        return '13MP + 2MP'
    elif 'Yuva 2' in product_name:
        return '13MP + 2MP'
    elif 'Yuva Pro' in product_name:
        return '48MP + 5MP + 2MP'
    elif 'Yuva' in product_name:
        return '13MP + 2MP'
    
    # Agni Series
    elif 'Agni 5G' in product_name:
        return '64MP + 8MP + 2MP'
    elif 'Agni 2' in product_name:
        return '50MP + 2MP'
    elif 'Agni' in product_name:
        return '64MP + 8MP + 2MP'
    
    # Z Series
    elif 'Z' in product_name:
        return '13MP + 2MP'
    
    # A Series
    elif 'A' in product_name:
        return '13MP + 2MP'
    
    # B Series
    elif 'B' in product_name:
        return '8MP + 2MP'
    
    # C Series
    elif 'C' in product_name:
        return '8MP + 2MP'
    
    else:
        return 'Unknown Camera'

df.loc[df['Camera'].isnull(), 'Camera'] = df.loc[df['Camera'].isnull(), 'Product Name'].apply(get_lava_camera)

In [9]:
df.isnull().sum()

Product Name       0
Current Price      0
MRP                0
Rating             0
Review text        6
RAM                0
Storage            0
Display            0
Camera             0
Battery            0
Processor          0
Product URL        0
Image URL          0
Discount_Number    0
Brand              0
Ratings            0
Reviews            0
dtype: int64

In [10]:
df.to_excel('Cleaned_lava.xlsx', index=False)